In [4]:
# minimal_check_ip_with_uc.py
import os
import re
import time
import undetected_chromedriver as uc

URL = "https://www.cman.jp/network/support/go_access.cgi"

# 環境変数 PROXY があれば使う。なければ固定値にフォールバック。
# 例: export PROXY=http://<your-squid-ip>:3128
PROXY = os.getenv("PROXY", "http://160.251.176.145:3128")

def main():
    opts = uc.ChromeOptions()
    opts.add_argument("--incognito")
    opts.add_argument("--disable-gpu")
    opts.add_argument(f"--proxy-server={PROXY}")

    # バージョンはお使いの Chrome に合わせて。未指定でも可。
    driver = uc.Chrome(version_main=138, options=opts)
    driver.set_page_load_timeout(30)

    try:
        driver.get(URL)
        time.sleep(2)

        html = driver.page_source
        # ページから IPv4 を雑に抽出（十分シンプル）
        m = re.search(r"(?:\d{1,3}\.){3}\d{1,3}", html)
        ip = m.group(0) if m else "NOT FOUND"

        # UA も参考に出す
        ua = driver.execute_script("return navigator.userAgent")

        print(f"[OK] Accessed via proxy: {PROXY}")
        print(f"[INFO] Detected IP on page: {ip}")
        print(f"[INFO] User-Agent: {ua}")

    finally:
        driver.quit()

if __name__ == "__main__":
    main()


[OK] Accessed via proxy: http://160.251.176.145:3128
[INFO] Detected IP on page: 160.251.176.145
[INFO] User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Safari/537.36
